# Filtering the universe (Prompt #9, Steps 1-8)

Every registered feature gets a row: kept or excluded and why (EXCLUDE_NON_CAUSAL, EXCLUDE_INVALID, EXCLUDE_MISSINGNESS, EXCLUDE_UNAVAILABLE, EXCLUDE_NEAR_CONSTANT, EXCLUDE_NUMERICAL, EXCLUDE_FAILED_NULL). Quality reads feature values only; the null screen reads development outcomes only. Drift is classified, never a reason to drop.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.selection` and `xauusd_quant.research.feature_selection_*` and read back from `results/feature_selection/`. Generate those first (needs the Prompt #8 research of the same timeframe for the pipeline-null veto):

```
xq feature-select --timeframe 5m                        # every stage + report + manifests
xq feature-select --timeframe 5m --target residual_reduction
xq feature-select --timeframe 5m --target future_return --horizon 5
xq feature-selection-report --timeframe 5m              # summary, SEL-H ledger rows, plots
xq build-feature-manifest --set standard --timeframe 5m # verify + reproduce the matrix
```

**Periods.** Development 2003-2017 selects; validation 2018-2021 only evaluates what development chose and reads the plateaus; the reserved test period (2022-) is never loaded with outcomes. Diagnostic models are linear (ridge, Lasso, elastic net, L1 logistic) on training-CDF rank designs - not the ML system of Prompt #10, and nothing here is a trading rule.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.selection.config import load_selection_config

scfg = load_selection_config(ROOT / "config" / "feature_selection.yaml")

TIMEFRAME = "5m"       # 5m | 15m (configured); 30m | 1h if they were run

out = scfg.results_path / TIMEFRAME
if not out.exists():
    print(f"No selection for {TIMEFRAME}. Run:  xq feature-select --timeframe {TIMEFRAME}")

def table(name):
    """Load a previously generated selection table (None if it was not written)."""
    path = out / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = out / "plots" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no plot {path.name})")

def load_json(name):
    path = out / name
    return json.loads(path.read_text()) if path.exists() else {}

summary = load_json("summary.json")
sets = load_json("sets.json")
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Periods and folds

In [ ]:
load_json('period_split.json')

## Exclusions

In [ ]:
summary.get('exclusion_reasons')

In [ ]:
filtering = table('filtering')
filtering.select('feature', 'family', 'reasons', 'kept_quality', 'in_selection_universe', 'kinds_passing_development',
                 'missing_development', 'missing_validation', 'missing_reserved_values', 'drift_class', 'prompt8_status')

## Drift classes (Step 8)

In [ ]:
filtering.group_by('drift_class').len()

## Development evidence: what passes, by target kind

In [ ]:
dev = table('development_evidence')
dev.filter(~pl.col('feature').str.starts_with('probe_')).group_by('kind').agg(pl.col('passes').sum().alias('passing_tests'),
          pl.col('feature').filter(pl.col('passes')).n_unique().alias('passing_features'))

## Noise probes through the same screen

In [ ]:
load_json('universe.json').get('probes_passing_development_null')